# 1. Preprocessing

Converting a complex clinical guideline PDF into a text format that's much more suitable for chunking and embedding.

In [ ]:
!pip install unstructured-client beautifulsoup4

In [ ]:
# import os
# from google.colab import userdata

# # Safely pull credentials from Colab's secret manager
# os.environ["UNSTRUCTURED_API_KEY"] = userdata.get('UNSTRUCTURED_API_KEY')
# os.environ["UNSTRUCTURED_API_URL"] = userdata.get('UNSTRUCTURED_API_URL')

## 1. Upload PDF in Colab

In [ ]:
# from google.colab import files

# print("Upload your clinical practice guideline PDF:")
# uploaded = files.upload()

# # This automatically grabs the filename of whatever you just uploaded
# pdf_path = list(uploaded.keys())[0]
# print(f"Successfully uploaded: {pdf_path}")

In [ ]:
# import os
# import re
# from bs4 import BeautifulSoup
# from unstructured_client import UnstructuredClient
# from unstructured_client.models import shared

# # ---------------------------------------------------------------------------
# # 1. INITIALIZATION & CONFIGURATION
# # ---------------------------------------------------------------------------
# UNSTRUCTURED_API_KEY = os.getenv("UNSTRUCTURED_API_KEY", "your_api_key_here")
# UNSTRUCTURED_API_URL = os.getenv("UNSTRUCTURED_API_URL", "your_api_url_here")

# client = UnstructuredClient(
#     api_key_auth=UNSTRUCTURED_API_KEY,
#     server_url=UNSTRUCTURED_API_URL
# )

# # ---------------------------------------------------------------------------
# # 2. HELPER FUNCTION: TABLE LINEARIZATION
# # ---------------------------------------------------------------------------
# def linearize_html_table(html_str: str, table_title: str = "Table Data") -> str:
#     """
#     Converts a 2D HTML table string into a series of 1D natural language sentences
#     (Table Linearization/Serialization) to preserve context for embeddings.
#     """
#     if not html_str:
#         return ""

#     soup = BeautifulSoup(html_str, 'html.parser')
#     rows = soup.find_all('tr')
#     if not rows:
#         return ""

#     # Extract headers (from <th> elements, or fallback to the first row's <td> elements)
#     header_row = rows[0]
#     headers = [th.get_text(strip=True) for th in header_row.find_all(['th', 'td'])]

#     # If headers are empty or non-existent, provide generic column placeholders
#     if not any(headers):
#         headers = [f"Column {i+1}" for i in range(len(header_row.find_all(['th', 'td'])))]

#     linearized_sentences = []

#     # Process data rows
#     for row in rows[1:]:
#         cells = row.find_all('td')
#         if not cells:
#             continue

#         # Treat the first column cell as the Row Identifier/Label
#         row_label = cells[0].get_text(strip=True)

#         row_statements = []
#         for idx, cell in enumerate(cells):
#             # Skip mapping the row identifier onto itself
#             if idx == 0:
#                 continue

#             cell_value = cell.get_text(strip=True)
#             # Ensure corresponding header exists for the cell index
#             column_header = headers[idx] if idx < len(headers) else f"Column {idx+1}"

#             # Construct a declarative natural language sentence if the cell is populated
#             if cell_value and column_header:
#                 statement = f"Regarding '{table_title}', for '{row_label}', the '{column_header}' is defined as: {cell_value}."
#                 row_statements.append(statement)

#         if row_statements:
#             linearized_sentences.append(" ".join(row_statements))

#     return "\n\n".join(linearized_sentences)

# # ---------------------------------------------------------------------------
# # 3. MAIN PIPELINE EXECUTION
# # ---------------------------------------------------------------------------
# # Make sure to include this import at the top of your block!
# from unstructured_client.models import operations, shared

# def parse_clinical_guideline(file_path: str) -> str:
#     """
#     Parses a clinical PDF, cleans artifacts, ignores front/end matter,
#     and linearizes complex tables into a continuous, clean text stream.
#     """
#     print(f"[1/4] Sending '{os.path.basename(file_path)}' to Unstructured Hi-Res API...")

#     with open(file_path, "rb") as f:
#         files = shared.Files(content=f.read(), file_name=os.path.basename(file_path))

#     # 1. Define the parameters
#     params = shared.PartitionParameters(
#         files=files,
#         strategy=shared.Strategy.HI_RES,       # Deep learning layout analysis
#         infer_table_structure=True,            # Crucial: Generates html matrix in metadata
#         chunking_strategy=None,                # Raw elements returned for custom handling
#         split_pdf_page=True
#     )

#     # 2. FIX: Wrap the parameters into a PartitionRequest object
#     req = operations.PartitionRequest(partition_parameters=params)

#     try:
#         # 3. Pass the wrapped request object to the API
#         response = client.general.partition(request=req)
#         elements = response.elements
#     except Exception as e:
#         print(f"API Error occurred: {e}")
#         return ""

#     print(f"[2/4] Successfully retrieved {len(elements)} document elements.")
#     print("[3/4] Filtering artifacts, excluding front/end matter, and linearizing tables...")

#     cleaned_corpus = []
#     exclude_remaining_document = False
#     last_seen_header = "General Guidelines"

#     # Define common regex patterns for stripping page numbers or running footers
#     footer_pattern = re.compile(r'^\s*(page\s*\d+|\d+\s*of\s*\d+|\bguideline\b.*?20\d{2})\s*$', re.IGNORECASE)

#     for el in elements:
#         category = el.get("type")
#         text = el.get("text", "").strip()
#         metadata = el.get("metadata", {})
#         page_num = metadata.get("page_number", 1)

#         # A. Page-Based Filtering (Exclude Title Page / TOC in the first 2 pages)
#         if page_num <= 8:
#             continue

#         # B. Strip running headers/footers or pure numerical page markers
#         if category in ["Header", "Footer"] or footer_pattern.match(text):
#             continue

#         # C. Track section headers to give tables local semantic context
#         if category == "Title":
#             last_seen_header = text

#         # D. Boundary-Based Slicing (The "Kill Switch" for End Matter)
#         # Add specific sections unique to your document
#         if category == "Title" and any(word in text.lower() for word in ["references", "bibliography", "contributors", "index", "acknowledgements", "disclosure statement", "source of funding"]):
#             exclude_remaining_document = True
#             print(f"--> Encountered end-matter section '{text}' on Page {page_num}. Activating truncation.")

#         if exclude_remaining_document:
#             continue

#         # E. Process Valid Narrative Elements
#         if category in ["Title", "NarrativeText", "ListItem"]:
#             cleaned_corpus.append(text)

#         # F. Process and Linearize Tabular Content
#         elif category == "Table":
#             html_table = metadata.get("text_as_html")

#             if html_table:
#                 # Convert 2D structure into natural language prose rows
#                 linearized_table_text = linearize_html_table(html_table, table_title=last_seen_header)
#                 if linearized_table_text:
#                     cleaned_corpus.append(linearized_table_text)
#             else:
#                 # Fallback if table object detection succeeded but structure parsing failed
#                 if text:
#                     cleaned_corpus.append(text)

#     print("[4/4] Pipeline processing complete.")

#     # Unify elements with double newlines
#     return "\n\n".join(cleaned_corpus)

# # ---------------------------------------------------------------------------
# # 4. RUNTIME ENTRYPOINT
# # ---------------------------------------------------------------------------
# if __name__ == "__main__":
#     # Example execution (Replace with your actual clinical guideline PDF file path)
#     pdf_path = "CPG_Intracranial_Haemorrhage.pdf"

#     if os.path.exists(pdf_path):
#         final_text_stream = parse_clinical_guideline(pdf_path)

#         # Verify the pipeline output snippet
#         print("\n--- Pipeline Sample Output Window ---")
#         print(final_text_stream[1000:]) # Print first 1200 characters

#         # This clean stream 'final_text_stream' is now ready to be passed directly
#         # to your fixed-size chunker script (Section 3.5.1).
#     else:
#         print(f"Please place a valid PDF file at '{pdf_path}' to execute the script.")

## Import output file

In [ ]:
# from google.colab import files

# # 1. Name your output text file
# output_filename = "linearized_cpg_text.txt"

# # 2. Save the final_text_stream string into a text file in Colab's temporary storage
# with open(output_filename, "w", encoding="utf-8") as f:
#     f.write(final_text_stream)

# print(f"Successfully saved text to Colab environment as '{output_filename}'.")
# print("Triggering download to your computer now...")

# # 3. Download the file directly to your local machine's 'Downloads' folder
# files.download(output_filename)

## Make 'linearized_cpg_text.txt' permanent in the colab for use

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

with open('/content/drive/MyDrive/RAG_ICH/data/processed/linearized_cpg_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:1000])  # Preview the first 1000 characters